# Mereska — TikTok / RedNote Dubbing (Kaggle)

Alur: **ingest → analisis visual → storytelling Indonesia → Edge TTS → word timing → subtitle animasi → MP4**. Seluruh video dan rasio `source` dipertahankan; canvas tambahan nonaktif secara default.

Versi ini memakai **Python Kaggle yang sedang berjalan dan source mode** seperti notebook dubbing yang berhasil. Tidak membuat virtual environment atau mewajibkan editable install.

Gemini dibatasi **3 percobaan total**, timeout request **30 detik**, dan anggaran **90 detik per tugas**. Jika fallback aktif dan layanan Gemini tetap gagal, tugas teks dialihkan ke Qwen lokal. Status fallback digunakan bersama oleh tahap dalam satu run; run baru mendapatkan identitas baru.

Sebelum menjalankan:
1. Aktifkan **Internet** dan GPU Kaggle untuk Qwen. `VISION_BACKEND` mengatur vision, sedangkan `TEXT_LLM_PROVIDER` mengatur penulisan teks.
2. Tambahkan Kaggle Secret **GOOGLE_API_KEY** bila vision atau teks memakai Gemini. Mode seluruhnya lokal tidak memerlukan key Gemini.
3. Konfirmasi hak penggunaan melalui `I_HAVE_RIGHTS = True`.
4. Gunakan checkout yang sudah memuat implementasi retry/fallback baru; cell setup memeriksa fitur sebelum memasang dependency.

Log Kaggle yang Anda kirim telah menunjukkan contoh RedNote berhasil diunduh dan dianalisis. **Revisi notebook ini belum dieksekusi atau diuji audiovisual.**


## 1. Konfigurasi

- `VIDEO_INPUT`: URL/share text TikTok, RedNote/Xiaohongshu, atau path video lokal di Kaggle.
- `ENABLE_CANVAS = False`: hanya `final_id_source.mp4`, komposisi asli dipertahankan.
- `ENABLE_CANVAS = True`: tambah `final_id_canvas_9x16.mp4` ber-background blur. Output source tetap dibuat. Ini mengatur canvas **visual tambahan**; scheduler audio tetap aktif agar suara sinkron.
- `NARRATION_SOURCE = "visual"`: cerita dari kejadian visual. Pilih `"audio"` untuk konteks dialog/subtitle/ASR; ini tetap narration pipeline, bukan terjemahan dialog kata demi kata.
- `START_FROM`: awalnya `"segment"`; setelah kegagalan dapat diubah ke `"analyze"`, `"narrate"`, `"dub"`, atau `"render"` jika artefak tahap sebelumnya tersedia.
- `PROJECT_NAME = ""`: otomatis berdasarkan identitas video. Gunakan nama baru untuk eksperimen terpisah.
- `TEXT_LLM_PROVIDER = "gemini"` dan `LOCAL_FALLBACK_ENABLED = True`: Gemini dahulu, Qwen lokal saat gagal. Pilih `"local"` untuk langsung memakai Qwen sebagai penulis.
- Batas `GEMINI_MAX_ATTEMPTS = 3` berbeda dari `FIT_RETRIES`: yang pertama membatasi panggilan LLM, yang kedua membatasi revisi naskah berdasarkan durasi suara.
- Untuk melanjutkan project pada log Anda, gunakan nama project existing (misalnya `story2u`) dan `START_FROM = "dub"` bila naskah/metadata/sumber masih tersedia. Re-run cell identitas/config dan ingest untuk memverifikasi cache.


In [ ]:
from pathlib import Path
import os
import sys
import json
import re
import subprocess
import shutil
import time

REPO_URL = "https://github.com/NaufalRizqullah/dubbingstory.git"
GIT_REF = "feat/rednote-only-dubbing"
WORK_ROOT = Path("/kaggle/working")
REPO_DIR = WORK_ROOT / "dubbingstory-social"
OUTPUT_ROOT = WORK_ROOT / "dubbingstory_social_outputs"

VIDEO_INPUT = "https://www.rednote.com/discovery/item/6ad206ee000000001400240a"
# Alternatif: "https://www.tiktok.com/@creator/video/..." atau "/kaggle/input/dataset/video.mp4"
I_HAVE_RIGHTS = False
PROJECT_NAME = ""
DOWNLOAD_QUALITY = "max"  # 1080 membatasi TINGGI; max menjaga sumber portrait 1080x1920.

NARRATION_SOURCE = "visual"  # visual | audio
STYLE = "viral_fb"          # viral_fb | jozzr_social | documentary | technical | calm_educational
VOICE_ID = "id-ID-GadisNeural"
NARRATION_FIT = "rewrite"   # rewrite | error; video tidak dipercepat/dipotong.
FIT_RETRIES = 2
SPEECH_COVERAGE = 0.85
ORIGINAL_VOLUME = 0.15
NARRATION_VOLUME = 1.0

SUBTITLE_STYLE = "karaoke"   # karaoke | word | classic
BURN_SUBTITLES = True
WORD_ALIGNMENT = "edge"     # edge | auto | whisperx; selain edge memasang extra alignment.
CAPTION_ANIMATION = "pop"   # pop | fade | scale | none
FONT_NAME = "DejaVu Sans"
FONT_PATH = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"
FONT_SIZE = 64
FONT_COLOR = "#FFFFFF"
ACTIVE_COLOR = "#FFE600"
OUTLINE_COLOR = "#000000"
OUTLINE_WIDTH = 3
CAPTION_POSITION = "bottom"
SAFE_MARGIN_X = 0.08
SAFE_MARGIN_Y = 0.18
MAX_WORDS = 5

ENABLE_CANVAS = False
FOREGROUND_ZOOM = 25.0
BACKGROUND_ZOOM = 100.0
BACKGROUND_BLUR = 50.0
BACKGROUND_GRAYSCALE = 1.0

MIN_SCENE_DURATION = 2.0
MAX_SCENE_DURATION = 12.0
SCENE_THRESHOLD = 3.0
MAX_KEYFRAMES = 5
START_FROM = "segment"

VISION_BACKEND = "qwen"     # qwen | gemini
MODEL_NAME = "Qwen/Qwen3-VL-4B-Instruct"
GEMINI_MODEL = "gemini-2.5-flash"
VLLM_VERSION = "0.27.1"     # Terikat pada preset wheel CUDA di cell setup.
VLLM_PARALLEL_MODE = "auto" # auto | data_parallel | tensor_parallel | single
GPU_MEMORY_UTILIZATION = 0.85
MAX_MODEL_LEN = 10240
VISION_MAX_TOKENS = 640
VISION_CONCURRENCY_OVERRIDE = None
IMAGE_MODE = "file"        # file | data; file diizinkan hanya di OUTPUT_ROOT.
PORT = 8000
SERVER_TIMEOUT_SECONDS = 900
WHISPER_MODEL = "base"     # Hanya dipakai jika NARRATION_SOURCE = audio.

# --- TEXT LLM / RETRY ---
TEXT_LLM_PROVIDER = "gemini"     # gemini | local; terpisah dari VISION_BACKEND.
LOCAL_FALLBACK_ENABLED = True
GEMINI_MAX_ATTEMPTS = 3          # Total panggilan, termasuk percobaan pertama.
GEMINI_TIMEOUT_SECONDS = 30
GEMINI_BUDGET_SECONDS = 90
GEMINI_INITIAL_WAIT_SECONDS = 5
GEMINI_WAIT_INCREMENT_SECONDS = 5
LOCAL_MAX_ATTEMPTS = 2
LOCAL_TIMEOUT_SECONDS = 60
LOCAL_BUDGET_SECONDS = 120
LOCAL_MAX_TOKENS = 2048
INSTALL_DEPENDENCIES = True     # False hanya jika runtime ini sudah disiapkan.

def needs_local_server(start_from, vision_backend, text_provider, fallback_enabled):
    vision_needed = start_from in {"segment", "analyze"} and vision_backend == "qwen"
    text_needed = start_from != "render" and (text_provider == "local" or fallback_enabled)
    return vision_needed or text_needed

def needs_gemini_secret(start_from, vision_backend, text_provider):
    vision_needed = start_from in {"segment", "analyze"} and vision_backend == "gemini"
    text_needed = start_from != "render" and text_provider == "gemini"
    return vision_needed or text_needed


## 2. Checkout dan setup source mode

Notebook memakai `sys.executable` dari kernel Kaggle dan menjalankan `main.py` langsung dari repository. Dependency dipasang tanpa editable build, tanpa bootstrap venv, dan tanpa `--upgrade` massal. Dependency project serta pin vLLM diselesaikan bersama agar tidak memasang Torch versi bebas lalu menggantinya lagi.

Checkout existing dipakai kembali tanpa pull/reset atau penghapusan working directory. Bila preflight menemukan kode lama, gunakan checkout baru yang memuat perubahan terbaru. `INSTALL_DEPENDENCIES = False` dapat dipakai untuk sesi yang sudah disiapkan; pemeriksaan runtime tetap berjalan.

`pip check` global Kaggle dapat melaporkan konflik paket bawaan yang tidak terkait pipeline. Hasilnya tetap ditampilkan dan disimpan; pemeriksaan import khusus pipeline dijalankan dari subprocess baru.

Preset vLLM 0.27.1 memilih wheel **cu130** untuk Torch 2.13.0, TorchVision 0.28.0, dan TorchAudio 2.11.0 dalam transaksi yang sama. TorchAudio 2.11 memakai stable ABI; nomor versinya tidak perlu disamakan dengan Torch. `pip check` saja tidak dapat mendeteksi CUDA 13.0 vs 12.8.

Sebelum ingest/startup Qwen, setup memvalidasi versi paket terhadap preset lalu wajib mengimpor TorchAudio, TorchVision, Transformers audio utilities, dan modul API server vLLM dalam subprocess baru, termasuk ketika `INSTALL_DEPENDENCIES=False`. Pemeriksaan ini tidak memuat model. Jika library GPU sudah pernah diimpor langsung ke kernel, restart kernel sebelum instalasi; pertahankan folder output project.


In [ ]:
if not Path("/kaggle").exists():
    raise RuntimeError("Notebook ini khusus Kaggle.")
if VISION_BACKEND not in {"qwen", "gemini"} or NARRATION_SOURCE not in {"visual", "audio"}:
    raise ValueError("Periksa VISION_BACKEND dan NARRATION_SOURCE.")
if TEXT_LLM_PROVIDER not in {"gemini", "local"}:
    raise ValueError("TEXT_LLM_PROVIDER harus gemini atau local.")
if WORD_ALIGNMENT not in {"edge", "auto", "whisperx"}:
    raise ValueError("WORD_ALIGNMENT harus edge, auto, atau whisperx.")
if SUBTITLE_STYLE not in {"classic", "word", "karaoke"} or NARRATION_FIT not in {"rewrite", "error"}:
    raise ValueError("Periksa SUBTITLE_STYLE dan NARRATION_FIT.")
if START_FROM not in {"segment", "analyze", "narrate", "dub", "render"}:
    raise ValueError("START_FROM tidak dikenal.")
if not isinstance(ENABLE_CANVAS, bool) or not isinstance(LOCAL_FALLBACK_ENABLED, bool):
    raise ValueError("Toggle canvas/fallback harus True atau False.")
if not I_HAVE_RIGHTS:
    raise ValueError("Konfirmasi hak penggunaan video melalui I_HAVE_RIGHTS sebelum menjalankan pipeline.")
for name, value in {
    "GEMINI_MAX_ATTEMPTS": GEMINI_MAX_ATTEMPTS, "LOCAL_MAX_ATTEMPTS": LOCAL_MAX_ATTEMPTS,
    "GEMINI_TIMEOUT_SECONDS": GEMINI_TIMEOUT_SECONDS, "GEMINI_BUDGET_SECONDS": GEMINI_BUDGET_SECONDS,
    "LOCAL_TIMEOUT_SECONDS": LOCAL_TIMEOUT_SECONDS, "LOCAL_BUDGET_SECONDS": LOCAL_BUDGET_SECONDS,
}.items():
    if not isinstance(value, (int, float)) or value <= 0:
        raise ValueError(f"{name} harus positif.")

WORK_ROOT.mkdir(parents=True, exist_ok=True)
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", GIT_REF, "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
elif not (REPO_DIR / ".git").is_dir():
    raise RuntimeError("REPO_DIR berisi folder non-Git. Pilih folder checkout baru.")
else:
    print("Memakai checkout existing; tidak melakukan pull/reset.")
REPO_REVISION = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print("Commit:", REPO_REVISION)

required_markers = {
    "dubbingstory/ingest/sources.py": ["normalize_source_url", '"tiktok"', '"xiaohongshu"', 'return "RedNote"'],
    "dubbingstory/ingest/rednote.py": ["class RedNoteIE", "XiaoHongShuIE"],
    "dubbingstory/ingest/youtube.py": ["_extract_source_info", "add_info_extractor(RedNoteIE())"],
    "dubbingstory/cli.py": ["--narration-source", "--narration-fit", "--subtitle-style", "--vertical-layout"],
    "dubbingstory/tts/edge_tts_engine.py": ["WordBoundary"],
    "dubbingstory/tts/voice_manager.py": ["strict_timing_validated", "atrim=end_sample", "Narration canvas:"],
    "dubbingstory/render/video_render.py": ["_canvas", "source_sha256"],
    "dubbingstory/render/audio_mix.py": ["channel_layouts=stereo", "sample_fmts=dbl"],
    "dubbingstory/llm/text_client.py": ["TextLLMClient"],
}
for relative, markers in required_markers.items():
    source_path = REPO_DIR / relative
    content = source_path.read_text(encoding="utf-8") if source_path.is_file() else ""
    if any(marker not in content for marker in markers):
        raise RuntimeError(f"Checkout belum memuat fitur terbaru: {relative}. Pakai checkout yang sudah membawa perbaikan RedNote, retry/fallback lokal, dan durasi canvas audio.")

if not shutil.which("ffmpeg") or not shutil.which("ffprobe") or not Path(FONT_PATH).is_file():
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg", "fonts-dejavu-core"], check=True)
if not Path(FONT_PATH).is_file():
    raise RuntimeError("FONT_PATH tidak ditemukan.")

RUNTIME_PYTHON = Path(sys.executable)
print("Python Kaggle:", RUNTIME_PYTHON, sys.version.split()[0])
pip_probe = subprocess.run([str(RUNTIME_PYTHON), "-m", "pip", "--version"], capture_output=True, text=True)
if pip_probe.returncode:
    raise RuntimeError("Interpreter kernel Kaggle ini tidak menyediakan pip. Pilih kernel/runtime Kaggle yang sudah bekerja.")
NEED_LOCAL_MODEL = needs_local_server(START_FROM, VISION_BACKEND, TEXT_LLM_PROVIDER, LOCAL_FALLBACK_ENABLED)
INSTALL_LOG = WORK_ROOT / "dubbingstory_social_install.log"

def setup_log_line(line):
    line = re.sub(r"(https?://)[^/@\s]+@", r"\1[redacted]@", str(line))
    return re.sub(r"(https?://[^\s?\"'<>]+)\?[^\s\"'<>]*", r"\1?[redacted]", line)

def pip_stage(arguments, label, required=True):
    print(f"\n[SETUP] {label}")
    with INSTALL_LOG.open("a", encoding="utf-8") as log:
        log.write(f"\n=== {label} ===\n")
        process = subprocess.Popen([str(RUNTIME_PYTHON), "-m", "pip", *arguments],
            cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding="utf-8", errors="replace", bufsize=1)
        try:
            for line in process.stdout:
                clean = setup_log_line(line)
                print(clean, end="", flush=True)
                log.write(clean)
            code = process.wait()
        finally:
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
    if code and required:
        raise RuntimeError(f"Setup gagal: {label}. Periksa {INSTALL_LOG.name}.")
    return code == 0

def local_runtime_install_spec(vllm_version):
    # vLLM 0.27.1 uses TorchAudio 2.11 stable ABI with Torch 2.13.
    # CUDA wheel variants must match even though base versions differ.
    if vllm_version != "0.27.1":
        raise ValueError("Preset CUDA ini untuk vLLM 0.27.1. Perbarui pasangan versi dan varian CUDA bersama sebelum mengganti VLLM_VERSION.")
    return (
        ["vllm==0.27.1", "torch==2.13.0+cu130", "torchvision==0.28.0+cu130", "torchaudio==2.11.0+cu130"],
        ["--extra-index-url", "https://download.pytorch.org/whl/cu130"],
    )

runtime_packages, runtime_index_args = local_runtime_install_spec(VLLM_VERSION) if NEED_LOCAL_MODEL else ([], [])
if NEED_LOCAL_MODEL and INSTALL_DEPENDENCIES:
    loaded_runtime = [name for name in ("torch", "torchaudio", "torchvision", "vllm") if name in sys.modules]
    if loaded_runtime:
        raise RuntimeError("Library GPU sudah diimpor ke kernel. Restart kernel (tanpa menghapus output project), lalu jalankan ulang setup sebelum mengganti wheel CUDA.")

if INSTALL_DEPENDENCIES:
    try:
        import tomllib
    except ImportError:
        raise RuntimeError("Preset Kaggle ini memerlukan Python 3.11+ untuk membaca pyproject.toml.") from None
    with (REPO_DIR / "pyproject.toml").open("rb") as source:
        project_metadata = tomllib.load(source)["project"]
    packages = list(project_metadata.get("dependencies", []))
    optional = project_metadata.get("optional-dependencies", {})
    packages += optional.get("openai-vision", [])
    packages += ["jsonschema>=4.21"]
    if WORD_ALIGNMENT != "edge" and SUBTITLE_STYLE != "classic":
        packages += optional.get("alignment", [])
    if NARRATION_SOURCE == "audio":
        packages.append("faster-whisper")
    packages += runtime_packages
    packages = list(dict.fromkeys(packages))
    pip_stage(["install", "--prefer-binary", *runtime_index_args, *packages], "Dependency project + runtime CUDA yang cocok dalam satu transaksi")
else:
    print("Instalasi dependency dilewati sesuai konfigurasi.")

if not pip_stage(["check"], "Pemeriksaan dependency global Kaggle", required=False):
    print("Ada konflik dependency pada environment global. Tinjau log; pemeriksaan import pipeline berikut tetap wajib.")

verify_code = """
import sys
from importlib.metadata import version
import dubbingstory
from dubbingstory.llm.text_client import TextLLMClient
from dubbingstory.ingest.sources import normalize_source_url
from dubbingstory.tts.edge_tts_engine import EdgeTTSEngine
from dubbingstory.vision.gemini_analyzer import GeminiVideoAnalyzer
from google.genai import types
types.GenerateContentConfig(http_options=types.HttpOptions(timeout=1000, retry_options=types.HttpRetryOptions(attempts=1)))
print('Source mode:', dubbingstory.__file__)
print('Runtime:', sys.version.split()[0])
print({p: version(p) for p in ('yt-dlp', 'edge-tts', 'google-genai', 'openai')})
"""
subprocess.run([str(RUNTIME_PYTHON), "-c", verify_code], cwd=REPO_DIR, check=True)
local_verify_code = """
import importlib
import json
import sys
from importlib.metadata import version
expected = json.loads(sys.argv[1])
actual = {package: version(package) for package in expected}
mismatches = {package: {"expected": expected[package], "installed": actual[package]}
              for package in expected if actual[package] != expected[package]}
if mismatches:
    raise RuntimeError("Runtime belum sesuai preset: " + json.dumps(mismatches))
import torch
import torchaudio
import torchvision
if torch.version.cuda != "13.0":
    raise RuntimeError("Preset vLLM 0.27.1 memerlukan wheel CUDA 13.0; periksa hasil instalasi.")
importlib.import_module("transformers.audio_utils")
importlib.import_module("vllm.entrypoints.openai.api_server")
print(json.dumps({
    "torch": torch.__version__, "torch_cuda": torch.version.cuda,
    "torchaudio": torchaudio.__version__, "torchvision": torchvision.__version__,
    "vllm": version("vllm"), "transformers": version("transformers"),
    "preflight": "imports passed; model not loaded",
}))
"""

def verify_local_runtime():
    # Import the failing path in a fresh process; never load a model here.
    print("[SETUP] Memeriksa import CUDA / TorchAudio / Transformers / vLLM...")
    expected_versions = dict(requirement.split("==", 1) for requirement in runtime_packages)
    try:
        probe = subprocess.run([str(RUNTIME_PYTHON), "-c", local_verify_code, json.dumps(expected_versions)],
            cwd=REPO_DIR, capture_output=True, text=True, encoding="utf-8",
            errors="replace", timeout=180)
    except subprocess.TimeoutExpired:
        raise RuntimeError("Preflight vLLM melebihi 180 detik. Periksa runtime Kaggle sebelum menjalankan pipeline.") from None
    output = setup_log_line((probe.stdout or "") + (probe.stderr or ""))
    with INSTALL_LOG.open("a", encoding="utf-8") as log:
        log.write("\n=== Local runtime import preflight ===\n" + output)
    if output:
        print(output, end="" if output.endswith("\n") else "\n")
    if probe.returncode:
        raise RuntimeError(
            "Preflight vLLM gagal sebelum model dimuat. Pastikan INSTALL_DEPENDENCIES=True, "
            "Torch/TorchVision/TorchAudio memakai varian cu130 yang cocok, lalu ulangi setup. "
            f"Detail tersimpan di {INSTALL_LOG.name}."
        )
    print("[SETUP] Import runtime lokal berhasil; model belum dimuat.")

if NEED_LOCAL_MODEL:
    verify_local_runtime()

filters = subprocess.run(["ffmpeg", "-hide_banner", "-filters"], capture_output=True, text=True, check=True)
if BURN_SUBTITLES and "subtitles" not in filters.stdout + filters.stderr:
    raise RuntimeError("FFmpeg tidak menyediakan filter subtitles/libass.")
print("Source mode siap. Belum memuat model atau menjalankan pipeline.")


## 3. Identitas sumber dan konfigurasi runtime

URL hanya diterima untuk TikTok atau RedNote/Xiaohongshu. File lokal tetap tersedia jika platform membatasi unduhan. Share text boleh berisi tepat satu URL. Normalisasi memakai adapter repository; query yang dibutuhkan extractor tidak dibuang.

Konfigurasi disimpan tanpa API key. Semua output berada di `/kaggle/working/dubbingstory_social_outputs/<project>/`.


In [ ]:
def render_options(enable_canvas):
    return {
        "ratios": ["source", "9:16"] if enable_canvas else ["source"],
        "vertical_layout": "layered" if enable_canvas else "pad",
    }

def expected_output_names(enable_canvas):
    names = ["final_id_source.mp4"]
    if enable_canvas:
        names.append("final_id_canvas_9x16.mp4")
    return names

def ingestion_arguments(video_input, is_url, project, quality, rights):
    if not rights:
        raise ValueError("Konfirmasi hak penggunaan video terlebih dahulu.")
    arguments = ["ingest", "--project", project, "--quality", quality]
    if is_url:
        arguments += ["--url", video_input, "--i-have-rights"]
    else:
        arguments += ["--input", video_input]
    return arguments

def pipeline_steps(project, start_from):
    # Ini jalur full existing: tidak memanggil summary/cut/concat.
    stages = [
        ("segment", ["segment", "--project", project]),
        ("analyze", ["analyze", "--project", project]),
        ("narrate", ["narrate", "--project", project, "--lang", "id"]),
        ("dub", ["dub", "--project", project, "--engine", "edge", "--voice-id", VOICE_ID]),
        ("render", ["render", "--project", project, "--mode", "full"]),
    ]
    names = [name for name, _ in stages]
    return stages[names.index(start_from):]

IS_URL = bool(re.search(r"https?://", VIDEO_INPUT, re.I))
if IS_URL:
    # Input melalui stdin, bukan log. Output hanya provider dan identity tanpa query.
    identity_code = """
import json, sys
from dubbingstory.ingest.sources import normalize_source_url, SourceIngestError
try:
    source = normalize_source_url(sys.stdin.read())
    if source.provider not in {"tiktok", "xiaohongshu"}:
        raise ValueError("Notebook ini khusus TikTok dan RedNote/Xiaohongshu.")
    print(json.dumps({"provider": source.provider, "project": source.identity.replace(":", "_")}))
except (SourceIngestError, ValueError) as exc:
    print(str(exc), file=sys.stderr)
    sys.exit(1)
"""
    identity_result = subprocess.run([str(RUNTIME_PYTHON), "-c", identity_code],
        input=VIDEO_INPUT, capture_output=True, text=True, cwd=REPO_DIR)
    if identity_result.returncode:
        raise RuntimeError(identity_result.stderr.strip() or "URL sumber tidak valid.")
    SOURCE_IDENTITY = json.loads(identity_result.stdout)
else:
    local_path = Path(VIDEO_INPUT).expanduser().resolve()
    if not local_path.is_file():
        raise FileNotFoundError("File lokal tidak ditemukan; isi path /kaggle/input/... yang benar.")
    import hashlib
    digest = hashlib.sha256()
    with local_path.open("rb") as media:
        for block in iter(lambda: media.read(1024 * 1024), b""):
            digest.update(block)
    VIDEO_INPUT = str(local_path)
    SOURCE_IDENTITY = {"provider": "local", "project": "local_" + digest.hexdigest()[:20]}

ACTIVE_PROJECT = PROJECT_NAME.strip() or SOURCE_IDENTITY["project"]
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,100}", ACTIVE_PROJECT):
    raise ValueError("PROJECT_NAME hanya boleh huruf, angka, underscore, dan tanda minus (maksimal 101 karakter).")
PROJECT_DIR = OUTPUT_ROOT / ACTIVE_PROJECT
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = PROJECT_DIR / "notebook_runtime.yaml"
BASE_URL = f"http://127.0.0.1:{PORT}/v1"

CONFIG = {
    "requested_languages": ["id"],  # Filter cached scripts/audio in standalone dub/render too.
    "project": {"output_dir": str(OUTPUT_ROOT)},
    "ingest": {"download_height": DOWNLOAD_QUALITY, "read_subtitles": NARRATION_SOURCE == "audio",
               "use_asr_fallback": NARRATION_SOURCE == "audio"},
    "whisper": {"model": WHISPER_MODEL, "device": "cpu", "compute_type": "int8"},
    "segment": {"min_scene_duration": MIN_SCENE_DURATION, "max_scene_duration": MAX_SCENE_DURATION,
                "adaptive_threshold": SCENE_THRESHOLD},
    "keyframes": {"max_per_scene": MAX_KEYFRAMES},
    "vision": {"provider": "openai" if VISION_BACKEND == "qwen" else "gemini",
               "gemini_model": GEMINI_MODEL, "openai_model": MODEL_NAME, "openai_base_url": BASE_URL,
               "openai_image_mode": IMAGE_MODE, "openai_max_tokens": VISION_MAX_TOKENS,
               "openai_model_max_context": MAX_MODEL_LEN, "concurrency": 1,
               "openai_structured_outputs": True, "openai_allow_schema_fallback": False},
    "llm": {
        "provider": TEXT_LLM_PROVIDER,
        "gemini": {"max_attempts": GEMINI_MAX_ATTEMPTS, "timeout_seconds": GEMINI_TIMEOUT_SECONDS,
                   "budget_seconds": GEMINI_BUDGET_SECONDS, "initial_wait_seconds": GEMINI_INITIAL_WAIT_SECONDS,
                   "wait_increment_seconds": GEMINI_WAIT_INCREMENT_SECONDS},
        "fallback": {"enabled": LOCAL_FALLBACK_ENABLED,
                     "state_path": str(PROJECT_DIR / "llm_runtime_state.json"), "run_id": ""},
        "local": {"base_url": BASE_URL, "model": MODEL_NAME, "max_attempts": LOCAL_MAX_ATTEMPTS,
                  "timeout_seconds": LOCAL_TIMEOUT_SECONDS, "budget_seconds": LOCAL_BUDGET_SECONDS,
                  "max_tokens": LOCAL_MAX_TOKENS, "context_window": MAX_MODEL_LEN},
    },
    "story": {"model": GEMINI_MODEL},
    "narration": {"languages": ["id"], "style": STYLE, "source": NARRATION_SOURCE,
                  "model": GEMINI_MODEL, "speech_coverage": SPEECH_COVERAGE},
    "tts": {"engine": "edge", "edge_voices": {"id": VOICE_ID}, "narration_fit": NARRATION_FIT,
            "fit_retries": FIT_RETRIES, "use_timeline_scheduler": True, "min_gap": 0.20},
    "render": {**render_options(ENABLE_CANVAS), "burn_subtitles": BURN_SUBTITLES,
               "audio_strategy": "dynamic_duck", "original_volume": ORIGINAL_VOLUME,
               "narration_volume": NARRATION_VOLUME, "foreground_zoom": FOREGROUND_ZOOM,
               "background_zoom": BACKGROUND_ZOOM, "background_blur": BACKGROUND_BLUR,
               "background_grayscale": BACKGROUND_GRAYSCALE},
    "subtitle": {"style": SUBTITLE_STYLE, "alignment": WORD_ALIGNMENT, "alignment_device": "cpu",
                 "font_name": FONT_NAME, "font_path": FONT_PATH, "font_size": FONT_SIZE,
                 "font_color": FONT_COLOR, "active_color": ACTIVE_COLOR, "outline_color": OUTLINE_COLOR,
                 "outline": OUTLINE_WIDTH, "position": CAPTION_POSITION, "animation": CAPTION_ANIMATION,
                 "safe_margin_x": SAFE_MARGIN_X, "safe_margin_y": SAFE_MARGIN_Y, "max_words": MAX_WORDS},
}

def save_config():
    # JSON merupakan subset YAML; loader repository membacanya sebagai YAML.
    CONFIG_PATH.write_text(json.dumps(CONFIG, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

save_config()
CHILD_ENV = os.environ.copy()
# Config notebook harus menjadi sumber pengaturan vision untuk subprocess.
for env_name in list(CHILD_ENV):
    if env_name.startswith("OPENAI_VISION_") or env_name in {"VISION_PROVIDER", "VISION_CONCURRENCY"}:
        CHILD_ENV.pop(env_name, None)
if VISION_BACKEND == "qwen":
    CHILD_ENV["OPENAI_VISION_API_KEY"] = "EMPTY"  # server lokal, bukan credential
CHILD_ENV["PYTHONUNBUFFERED"] = "1"
CHILD_ENV["PYTHON_DOTENV_DISABLED"] = "1"  # Kaggle Secrets/config, bukan .env lama.

def safe_log(text):
    # Jangan menampilkan query share URL atau credential yang diberikan ke child.
    text = re.sub(r"(https?://[^\s?\"'<>]+)\?[^\s\"'<>]*", r"\1?[redacted]", str(text))
    for name in ("GOOGLE_API_KEY", "HF_TOKEN", "HF_API_KEY", "OPENAI_VISION_API_KEY"):
        value = CHILD_ENV.get(name, "")
        if value and value != "EMPTY":
            text = text.replace(value, "[redacted]")
    return text

def run_cli(arguments, label):
    command = [str(RUNTIME_PYTHON), "-u", str(REPO_DIR / "main.py"), "--config", str(CONFIG_PATH), *arguments]
    log_path = PROJECT_DIR / f"notebook_{label}.log"
    print(f"\n▶ {label} | project={ACTIVE_PROJECT}")
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, cwd=REPO_DIR, env=CHILD_ENV,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                clean = safe_log(line)
                print(clean, end="", flush=True)
                log.write(clean)
            returncode = process.wait()
        finally:
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
    if returncode:
        raise RuntimeError(f"Tahap {label} gagal (exit {returncode}). Periksa {log_path.name}; jangan lanjut ke tahap berikutnya.")

print("Provider:", SOURCE_IDENTITY["provider"], "| Project:", ACTIVE_PROJECT)
print("Output:", ", ".join(expected_output_names(ENABLE_CANVAS)))
print("Canvas tambahan:", ENABLE_CANVAS, "| Konteks narasi:", NARRATION_SOURCE)
print("Text LLM:", TEXT_LLM_PROVIDER, "| Gemini attempts:", GEMINI_MAX_ATTEMPTS,
      "| Local fallback:", LOCAL_FALLBACK_ENABLED)


## 4. Ingest sumber — sebelum memuat Qwen

Adapter mempertahankan domain `rednote.com` untuk URL RedNote dan memakai parsing yt-dlp Xiaohongshu yang sama. Jangan mengganti domain menjadi `xiaohongshu.com`: pada pemeriksaan metadata publik, URL contoh RedNote menyediakan video sementara alamat Xiaohongshu untuk ID yang sama mengarah ke halaman error tanpa data video. Share link TikTok tetap memakai resolver existing. Source disimpan sekali bersama identitas dan hash, lalu tahap berikutnya menggunakan project tersebut. Jangan mengganti URL remote menjadi `--input source.mp4` karena itu dapat mengubah provenance.

Jika log masih menampilkan `Source: xiaohongshu` dengan URL `https://www.xiaohongshu.com/...` padahal `VIDEO_INPUT` memakai `rednote.com`, checkout kemungkinan masih memakai adapter lama. Label provider `xiaohongshu` tetap benar; yang harus dipertahankan adalah host URL RedNote. Notebook baru memeriksa adapter ini sebelum setup. Checkout existing tidak di-pull/reset otomatis; sinkronkan kode perbaikan atau pilih checkout baru yang memuatnya.

Bila run sebelumnya sudah menyimpan `source.mp4` dan metadata valid, pakai `PROJECT_NAME` serta `OUTPUT_ROOT` yang sama (misalnya project `story2u`) untuk reuse cache. `PROJECT_NAME=""` memilih nama otomatis, sehingga mungkin menunjuk folder lain dan mengunduh ulang.

Jika unduhan dibatasi, privat, tidak tersedia, atau extractor gagal: **berhenti**, lalu gunakan video lokal yang Anda berhak gunakan melalui `VIDEO_INPUT` dan project baru. Notebook tidak melewati login, CAPTCHA, DRM, atau proteksi platform. Log Kaggle yang Anda berikan menunjukkan link contoh berhasil diunduh (720×1280, sekitar 22,17 detik); ketersediaan platform pada run berikutnya tetap diperiksa saat ingest.


In [ ]:
run_cli(ingestion_arguments(VIDEO_INPUT, IS_URL, ACTIVE_PROJECT, DOWNLOAD_QUALITY, I_HAVE_RIGHTS), "ingest")
manifest_path = PROJECT_DIR / "ingest_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
metadata = json.loads((PROJECT_DIR / "video_metadata.json").read_text(encoding="utf-8"))
if manifest.get("status") != "ingested" or not (PROJECT_DIR / "source.mp4").is_file():
    raise RuntimeError("Ingest belum lengkap.")
print("Sumber siap:", metadata.get("provider"), "| Resolusi tampilan:",
      metadata.get("display_width"), "x", metadata.get("display_height"),
      "| Durasi video:", metadata.get("video_duration"))
if NARRATION_SOURCE == "audio" and not manifest.get("subtitle_data"):
    raise RuntimeError("Mode audio tidak mendapatkan transcript/subtitle. Periksa ASR, atau pilih mode visual secara eksplisit.")


## 5. Gemini Secret bila diperlukan

Key diambil dari Kaggle Secrets dan diteruskan melalui environment subprocess, tanpa ditulis ke file atau command line. Jika `TEXT_LLM_PROVIDER="local"` dan vision juga Qwen, cell melewati pengambilan key. Render-only juga tidak memerlukan Gemini.

Jika `VISION_BACKEND="gemini"`, tahap analisis tetap memerlukan key walaupun penulis teks memakai lokal. Fallback lokal ini berlaku pada tugas **teks**, bukan pengganti visual Gemini.


In [ ]:
if not needs_gemini_secret(START_FROM, VISION_BACKEND, TEXT_LLM_PROVIDER):
    CHILD_ENV.pop("GOOGLE_API_KEY", None)
    print("Tahap/provider yang dipilih tidak memerlukan Gemini Secret.")
else:
    from kaggle_secrets import UserSecretsClient
    try:
        gemini_key = UserSecretsClient().get_secret("GOOGLE_API_KEY").strip()
        if not gemini_key:
            raise ValueError("empty")
    except Exception:
        raise RuntimeError("Aktifkan Kaggle Secret GOOGLE_API_KEY untuk provider Gemini yang dipilih.") from None
    CHILD_ENV["GOOGLE_API_KEY"] = gemini_key
    del gemini_key
    print("Gemini Secret tersedia; nilainya tidak ditampilkan.")


## 6. Jalankan dubbing full dengan fallback

Scene detection/keyframe hanya untuk analisis. Narasi dan rewrite TTS tetap mengikuti timeline sumber. Batas Gemini mencakup pemanggilan pertama; rewrite naskah tidak boleh memulai rangkaian retry yang tak terbatas.

Server Qwen tersedia selama tahap yang membutuhkan vision lokal atau fallback teks, **termasuk resume langsung dari narrate/dub**. Server dihentikan sebelum render dan pada jalur kegagalan. Mode vision Gemini dengan fallback teks aktif juga memuat Qwen untuk penulisan.

Satu eksekusi cell mendapatkan `run_id` baru yang dibagikan melalui YAML ke semua subprocess. Status fallback disimpan dalam `llm_runtime_state.json` tanpa prompt/key, sehingga perpindahan ke lokal berlaku lintas tahap dalam run yang sama. Menjalankan ulang cell memulai run baru; tidak menghapus artefak video/naskah.

Boundary Edge tetap harus mencakup semua kata. Hasil lokal masih diperiksa struktur JSON, bahasa, referensi scene/claim, dan durasinya; kegagalan tidak diganti dengan timing kata perkiraan. Review kesesuaian cerita dengan video tetap diperlukan.

**Pemulihan error durasi canvas pada dub:** perbarui kode repository yang dipakai `REPO_DIR` terlebih dahulu; notebook baru tidak melakukan pull otomatis. Setup menolak renderer lama. Pertahankan `PROJECT_NAME`/`OUTPUT_ROOT`, pilih `START_FROM="dub"` jika source, storyboard, `scripts/script_id.json` dan `.txt` masih ada, lalu jalankan kembali setup/configuration/ingest dan pipeline. Analisis/narrate tidak diulang; TTS/fitting diulang karena hasil sebelumnya belum lengkap. Jangan langsung memilih render. Log baru mencatat `expected`, `actual`, dan `delta` durasi canvas. Guard 20 ms tetap aktif.


In [ ]:
import signal
import socket
import urllib.request
import uuid

def stop_server(process):
    if process is None:
        return
    try:
        os.killpg(process.pid, signal.SIGTERM)
    except ProcessLookupError:
        pass
    try:
        process.wait(timeout=20)
    except subprocess.TimeoutExpired:
        pass
    try:
        os.killpg(process.pid, signal.SIGKILL)
    except ProcessLookupError:
        pass
    process.wait()

def qwen_command():
    gpu_probe = subprocess.run([str(RUNTIME_PYTHON), "-c",
        "import torch,json; print(json.dumps([{'name':torch.cuda.get_device_name(i),"
        "'gb':round(torch.cuda.get_device_properties(i).total_memory/1024**3,1),"
        "'cc':list(torch.cuda.get_device_capability(i))} for i in range(torch.cuda.device_count())]))"],
        capture_output=True, text=True, env=CHILD_ENV, check=False)
    if gpu_probe.returncode:
        raise RuntimeError("Runtime PyTorch tidak dapat membaca GPU. Periksa log setup dan kompatibilitas vLLM/driver.")
    gpus = json.loads(gpu_probe.stdout)
    if not gpus:
        raise RuntimeError("GPU tidak ditemukan. Aktifkan GPU untuk Qwen, atau gunakan vision/teks Gemini dengan fallback lokal dimatikan.")
    if min(gpu["cc"][0] for gpu in gpus) < 7:
        raise RuntimeError("GPU ini terlalu lama untuk preset vLLM.")
    mode = VLLM_PARALLEL_MODE
    params = re.search(r"-(\d+(?:\.\d+)?)B(?:-|$)", MODEL_NAME, re.I)
    if mode == "auto":
        mode = "data_parallel" if len(gpus) > 1 and params and float(params.group(1)) <= 4.5 else (
            "tensor_parallel" if len(gpus) > 1 else "single")
    if mode not in {"single", "data_parallel", "tensor_parallel"}:
        raise ValueError("VLLM_PARALLEL_MODE tidak dikenal.")
    dp = len(gpus) if mode == "data_parallel" else 1
    tp = len(gpus) if mode == "tensor_parallel" else 1
    concurrency = dp if mode == "data_parallel" else min(2, tp)
    if VISION_CONCURRENCY_OVERRIDE is not None:
        concurrency = max(1, int(VISION_CONCURRENCY_OVERRIDE))
    CONFIG["vision"]["concurrency"] = concurrency
    save_config()
    command = [str(RUNTIME_PYTHON), "-m", "vllm.entrypoints.openai.api_server",
        "--model", MODEL_NAME, "--host", "127.0.0.1", "--port", str(PORT),
        "--max-model-len", str(MAX_MODEL_LEN), "--tensor-parallel-size", str(tp),
        "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION), "--dtype", "half",
        "--generation-config", "vllm", "--enforce-eager"]
    if dp > 1:
        command += ["--data-parallel-size", str(dp)]
    if IMAGE_MODE == "file":
        command += ["--allowed-local-media-path", str(OUTPUT_ROOT)]
    print("GPU:", gpus, "| Parallel mode:", mode, "| Vision concurrency:", concurrency)
    return command

def start_qwen_server():
    with socket.socket() as probe:
        if probe.connect_ex(("127.0.0.1", PORT)) == 0:
            raise RuntimeError("PORT sudah dipakai. Hentikan server lama atau pilih PORT lain.")
    command = qwen_command()
    server_env = CHILD_ENV.copy()
    server_env.pop("GOOGLE_API_KEY", None)
    server_env.pop("OPENAI_VISION_API_KEY", None)
    server_log = (PROJECT_DIR / "vllm_server.log").open("w", encoding="utf-8")
    process = None
    try:
        process = subprocess.Popen(command, cwd=REPO_DIR, env=server_env,
            stdout=server_log, stderr=subprocess.STDOUT, start_new_session=True)
        deadline = time.monotonic() + SERVER_TIMEOUT_SECONDS
        while time.monotonic() < deadline:
            if process.poll() is not None:
                raise RuntimeError("vLLM berhenti sebelum siap. Periksa vllm_server.log.")
            try:
                with urllib.request.urlopen(f"{BASE_URL}/models", timeout=5) as response:
                    models = json.load(response).get("data", [])
                if any(model.get("id") == MODEL_NAME for model in models):
                    return process, server_log
            except (OSError, ValueError):
                pass
            time.sleep(5)
        raise RuntimeError("vLLM belum siap sebelum timeout. Periksa vllm_server.log.")
    except BaseException:
        stop_server(process)
        server_log.close()
        raise

steps = pipeline_steps(ACTIVE_PROJECT, START_FROM)
required_artifact = {
    "segment": "source.mp4", "analyze": "segment_manifest.json",
    "narrate": "storyboard.json", "dub": "scripts/script_id.json",
    "render": "audio/audio_id.manifest.json",
}[START_FROM]
if not (PROJECT_DIR / required_artifact).is_file():
    raise RuntimeError(f"START_FROM={START_FROM} memerlukan {required_artifact}. Jalankan tahap sebelumnya.")

# Satu identitas run untuk semua proses CLI; breaker dari run lama tidak diwarisi.
CONFIG["llm"]["fallback"]["run_id"] = str(uuid.uuid4())
save_config()
print("Run:", CONFIG["llm"]["fallback"]["run_id"])
server_process, server_log = None, None
try:
    for stage, arguments in steps:
        stage_needs_local = stage in {"analyze", "narrate", "dub"} and needs_local_server(
            stage, VISION_BACKEND, TEXT_LLM_PROVIDER, LOCAL_FALLBACK_ENABLED)
        if stage_needs_local and server_process is None:
            server_process, server_log = start_qwen_server()
        elif not stage_needs_local and server_process is not None:
            stop_server(server_process)
            server_log.close()
            server_process, server_log = None, None
        run_cli(arguments, stage)
finally:
    stop_server(server_process)
    if server_log is not None:
        server_log.close()
print("\nPipeline full selesai. Periksa manifest dan preview pada cell berikutnya.")


## 7. Hasil dan file download

Preview hanya menampilkan output yang diminta konfigurasi saat ini, sehingga canvas lama tidak tampil ketika toggle dimatikan. ZIP berisi MP4 final, voice-over WAV, subtitle, naskah final, word timing, dan manifest; video sumber serta model tidak ikut dibundel.

Pemeriksaan visual tetap diperlukan: frame awal/akhir dan komposisi sama, narasi tidak terpotong, kata mengikuti suara, safe area tidak menutupi objek penting, serta ducking pulih saat narator berhenti.

ZIP dibuat sebelum preview. Kegagalan preview ditampilkan sebagai pemberitahuan dan tidak membatalkan hasil render atau bundle. Jika hanya tahap 7 yang gagal, ganti cell tahap 7 dengan versi ini dan jalankan ulang cell tersebut; tahap 6 tidak perlu diulang selama variabel sesi dan hasil masih tersedia.


In [ ]:
from IPython.display import display, Video, FileLink
import zipfile

render_manifest = json.loads((PROJECT_DIR / "render_manifest.json").read_text(encoding="utf-8"))
if render_manifest.get("status") != "complete":
    raise RuntimeError("Render belum lengkap. Periksa render_manifest.json dan log render.")
expected = [PROJECT_DIR / name for name in expected_output_names(ENABLE_CANVAS)]
reported = {Path(path).resolve() for path in render_manifest.get("rendered_videos", [])}
if any(not path.is_file() or path.resolve() not in reported for path in expected):
    raise RuntimeError("Output yang diminta tidak cocok dengan render_manifest. Jalankan render ulang.")

# Selesaikan ZIP sebelum preview opsional agar kegagalan display tidak menghalangi download.
bundle_path = WORK_ROOT / f"{ACTIVE_PROJECT}_dubbing_results.zip"
artifacts = expected + [
    PROJECT_DIR / "scripts" / f"script_id{suffix}"
    for suffix in (".txt", ".json", ".srt", ".words.json", ".ass")
] + [
    PROJECT_DIR / "ingest_manifest.json", PROJECT_DIR / "video_metadata.json",
    PROJECT_DIR / "audio" / "audio_id.wav",
    PROJECT_DIR / "audio" / "audio_id.manifest.json", PROJECT_DIR / "render_manifest.json",
    CONFIG_PATH, PROJECT_DIR / "llm_runtime_state.json",
]
artifacts += sorted((PROJECT_DIR / "scripts").glob("script_id.*.ass"))
with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in dict.fromkeys(artifacts):
        if path.is_file():
            archive.write(path, arcname=str(path.relative_to(PROJECT_DIR)))
# FileLink relatif ke /kaggle/working agar bisa diunduh melalui notebook Kaggle.
os.chdir(WORK_ROOT)
print("Bundle hasil:")
display(FileLink(bundle_path.name))
for path in expected:
    relative = path.relative_to(WORK_ROOT)
    display(FileLink(str(relative)))
    try:
        # Positional path also works on IPython versions that inspect data
        # before filename and otherwise call os.path.exists(None).
        display(Video(str(path), embed=True, width=360))
    except Exception as exc:
        print(f"Preview {path.name} tidak tersedia ({type(exc).__name__}). MP4 dan ZIP tetap dapat diunduh.")
print("Durasi sumber dan sinkronisasi caption tetap perlu diperiksa secara audiovisual.")


## 8. Opsional — render ulang tanpa vision/TTS

Ubah toggle/style di cell berikutnya, lalu jalankan. Memilih canvas mengubah komposisi **output tambahan** saja. Karaoke/word membutuhkan file timing yang sudah dibuat; jika awalnya memakai classic, ulangi `dub` dengan animated style lebih dulu.

Untuk perubahan naskah atau voice, ulangi tahap `dub` dan `render`, bukan hanya cell render ulang. File dengan nama output sama dapat diganti.

Untuk video baru, `BURN_SUBTITLES=False` pada konfigurasi mematikan subtitle yang dibakar ke gambar. SRT/ASS tetap bisa tersedia sebagai file terpisah. Untuk hasil yang sudah dibuat, set `RUN_RENDER_AGAIN=True` dan `RERENDER_BURN_SUBTITLES=False` pada cell di bawah, lalu jalankan cell render ulang dan tahap 7. Render ulang memakai sumber asli serta WAV existing, tanpa vision/TTS; MP4 dengan nama yang sama diganti. Subtitle yang sudah berada di video sumber tetap terlihat.


In [ ]:
RUN_RENDER_AGAIN = False
RERENDER_CANVAS = ENABLE_CANVAS
RERENDER_SUBTITLE_STYLE = SUBTITLE_STYLE
RERENDER_BURN_SUBTITLES = BURN_SUBTITLES  # False untuk output tanpa subtitle yang dibakar.

if RUN_RENDER_AGAIN:
    if RERENDER_SUBTITLE_STYLE not in {"classic", "word", "karaoke"}:
        raise ValueError("Style tidak dikenal.")
    ENABLE_CANVAS = bool(RERENDER_CANVAS)
    CONFIG["render"].update(render_options(ENABLE_CANVAS), burn_subtitles=bool(RERENDER_BURN_SUBTITLES))
    CONFIG["subtitle"]["style"] = RERENDER_SUBTITLE_STYLE
    save_config()
    run_cli(["render", "--project", ACTIVE_PROJECT, "--mode", "full"], "render")
    print("Jalankan kembali cell hasil untuk memperbarui preview dan ZIP.")
else:
    print("Render ulang dilewati.")


## Pengujian dan catatan runtime

Tests tidak berjalan otomatis. Jalankan di **Kaggle** menggunakan interpreter kernel yang sama, setelah checkout memuat perubahan ini:

```python
subprocess.run([sys.executable, "-m", "pip", "install", "pytest>=8,<10", "jsonschema>=4.21"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], cwd=REPO_DIR, check=True)
```

- Log sampel sebelumnya menunjukkan source RedNote, Qwen, dan penulisan naskah berhasil; bottleneck berikutnya adalah 429/503 Gemini saat fitting TTS. Angka waktu dari log tersebut bukan hasil pengujian revisi ini.
- `GEMINI_MAX_ATTEMPTS=3` adalah jumlah total, bukan tiga retry tambahan. `GEMINI_BUDGET_SECONDS` membatasi penjadwalan retry satu tugas termasuk jeda. Timeout transport bukan hard wall-clock deadline; startup model, upload video Gemini, TTS dan render berada di luar budget ini. Local fallback juga dibatasi.
- `FIT_RETRIES` membatasi berapa kali naskah direvisi dan diukur ulang. Ini berbeda dari jumlah percobaan API untuk satu revisi.
- Layanan Gemini bermasalah dapat membuka breaker untuk sisa run; request teks berikutnya memakai lokal. Run baru mempunyai ID baru dan boleh mencoba Gemini lagi.
- `TEXT_LLM_PROVIDER="local"` langsung memakai Qwen. Dengan vision Qwen juga, tidak perlu Secret Gemini. Kualitas bahasa Indonesia dan kesesuaian visual tetap perlu diperiksa.
- `START_FROM="dub"` dapat melanjutkan naskah existing tanpa mengulang analisis; Qwen tetap dimuat bila fallback aktif. `START_FROM="render"` tidak memuat LLM.
- Install menggunakan Python kernel dan source mode. Tidak ada venv, `ensurepip`, atau editable build. Dependency sudah memenuhi versi tidak di-upgrade massal. Simpan versi/commit dari log untuk reproduksi.
- Qwen OOM: kurangi `MAX_MODEL_LEN`, pakai model lebih kecil yang didukung, atau gunakan Gemini untuk vision dan teks dengan `LOCAL_FALLBACK_ENABLED=False`. Mengurangi konteks dapat membuat prompt ditolak. Router lokal tidak memotong prompt yang sudah dibangun; pembuat prompt existing masih merangkum sebagian evidence.
- Download ditolak platform: gunakan video lokal yang Anda berhak gunakan. Tidak ada login/DRM/CAPTCHA bypass.
- Burn subtitle memerlukan libass dan encoding video. Fitting narasi tidak memotong atau mempercepat video sumber.
- Jangan menyunting timestamp kata untuk menebak sinkronisasi. Koreksi teks ucapan atau pilih alignment opsional lalu ulangi dub.
- Tinjau output audiovisual dan jangan membagikan log/metadata tanpa memeriksa isinya.

Referensi: `dubbingstory/llm/text_client.py`, `dubbingstory/vision/gemini_analyzer.py`, `dubbingstory/story/gemini_json.py`, `dubbingstory/story/script_writer.py`, dan `docs/native-storytelling.md`.

Jika muncul error `PyTorch CUDA 13.0` vs `TorchAudio CUDA 12.8`, gunakan notebook ini dengan `INSTALL_DEPENDENCIES=True` dan ulangi setup. Jangan menghapus TorchAudio atau memaksa nomor versinya sama dengan Torch. Pertahankan `PROJECT_NAME`/`OUTPUT_ROOT` agar source yang sudah berhasil diunduh tetap dipakai. Jika segment manifest sudah tersedia, lanjutkan dengan `START_FROM="analyze"`.

Referensi runtime: [vLLM 0.27.1 CUDA requirements](https://github.com/vllm-project/vllm/blob/v0.27.1/requirements/cuda.txt), [kompatibilitas TorchAudio](https://docs.pytorch.org/audio/stable/installation.html), dan [wheel resmi cu130](https://download.pytorch.org/whl/cu130/torchaudio/).
